# ChagaSight — 1D Pathway Ablation  (`per_1d_fold.ipynb`)

### Purpose
Train **1D ViT FM only** (no 2D images, no REPA) on one fold.

### Fair-comparison checklist — must match hybrid exactly
| Criterion | This notebook | 2D notebook (`per_2d_fold.ipynb`) |
|-----------|--------------|----------------------------------|
| Fold | `FOLD = 0` | same |
| Loss | `AsymmetricBCE` γ⁺=0,γ⁻=2,w=10 | same |
| Eff.batch | 32 (16 × accum=2) | same |
| P1 iters | 2 000 (backbone frozen) | same |
| P2 iters | 12 000 (full) | same |
| Val every | 500 iters | same |
| Pretrain | ST-MEM (`stmem_1d_pretrained.pt`) | MAE (`mae_2d_pretrained.pt`) |
| Demographics | age + sex ✓ | not used |
| Sampler | WeightedRandom 5× | same |

**No alignment loss** — there is no 2D pathway to align to.

In [ ]:
import sys, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm import tqdm

# ── USER CONFIG ──────────────────────────────────────────────────────────────
FOLD          = 0        # use the SAME fold as per_2d_fold.ipynb
BATCH_SIZE    = 16
GRAD_ACCUM    = 2        # eff. batch = 32
PHASE1_ITERS  = 2_000    # backbone frozen  (head + demo encoder only)
PHASE2_ITERS  = 12_000   # full fine-tune  (matches per_2d_fold.ipynb for fair ablation)
VAL_EVERY     = 500
PHASE1_LR     = 2e-4
PHASE2_LR     = 2e-5
MAX_GRAD_NORM = 1.0
NUM_WORKERS   = 2
SEED          = 42

# ── PATHS ────────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path(r"D:\IIT\L6\FYP\ChagaSight")
METADATA_CSV  = PROJECT_ROOT / "data/processed/metadata/combined_5fold.csv"
IMAGES_DIR    = PROJECT_ROOT / "data/processed/2d_images"       # still needed by ChagasDataset
SIGNALS_DIR   = PROJECT_ROOT / "data/processed/1d_signals_100hz"
STMEM_CKPT    = PROJECT_ROOT / "checkpoints/stmem_1d_pretrained.pt"
CKPT_DIR      = PROJECT_ROOT / "checkpoints"
BEST_PATH     = CKPT_DIR / f"fold{FOLD}_1d_best.pt"
OUT_CSV       = CKPT_DIR / f"fold{FOLD}_1d_results.csv"

CKPT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Device : {device}")
print(f"Fold   : {FOLD}  |  Eff.batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"P1 iters: {PHASE1_ITERS}  |  P2 iters: {PHASE2_ITERS}  |  Val every: {VAL_EVERY}")
for p, name in [(STMEM_CKPT, "ST-MEM ckpt"), (METADATA_CSV, "metadata")]:
    print(f"{name}: {'✓' if p.exists() else '✗  MISSING: ' + str(p)}")

In [ ]:
def compute_challenge_score(labels, outputs,
                             fraction_capacity=0.05,
                             num_permutations=10_000, seed=12345):
    labels  = np.asarray(labels,  dtype=np.float64)
    outputs = np.asarray(outputs, dtype=np.float64)
    capacity = int(fraction_capacity * len(labels))
    np.random.seed(seed)
    tp = np.zeros(num_permutations)
    for i in range(num_permutations):
        idx     = np.random.permutation(len(labels))
        ordered = labels[idx][np.argsort(outputs[idx])[::-1]]
        tp[i]   = ordered[:capacity].sum()
    tp_mean = tp.mean()
    return float(tp_mean / (tp_mean + (labels.sum() - tp_mean) + 1e-8))

print("✓ Scorer loaded")


In [ ]:
from src.training.dataset import ChagasDataset, custom_collate_fn

# Same ChagasDataset as hybrid — loads both image and signal.
# The 1D-only model will simply ignore batch['image'].
train_ds = ChagasDataset(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),
    signals_dir=str(SIGNALS_DIR),
    split='train', fold=FOLD,
    augment=True, use_soft_labels=True,
)
val_ds = ChagasDataset(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),
    signals_dir=str(SIGNALS_DIR),
    split='val', fold=FOLD,
    augment=False, use_soft_labels=True,
)

sample_weights = train_ds.get_sample_weights()
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          drop_last=True, collate_fn=custom_collate_fn)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=custom_collate_fn)

print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")
print(f"Val pos: {val_ds.df['label_hard'].sum():,}")

In [ ]:
from src.models.vit_1d_fm import ViT1D_FM
from src.training.losses import AsymmetricBCELoss

class ViT1DClassifier(nn.Module):
    """1D FM backbone (+ demographics) + single-pathway head."""

    def __init__(self):
        super().__init__()
        self.backbone = ViT1D_FM(
            num_leads=12, seq_len=1000, patch_size=50,
            embed_dim=768, depth=12, num_heads=12,
            mlp_ratio=4.0, dropout=0.1,
            use_aol=True,
            use_demographics=True,   # age + sex modulation
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def freeze_backbone(self):
        # Freeze transformer layers and patch embed; keep demo encoder trainable
        # so Phase 1 trains: demo_encoder + head  (head is random-init, demos are cheap)
        for name, p in self.backbone.named_parameters():
            if 'demo_encoder' in name:
                p.requires_grad_(True)
            else:
                p.requires_grad_(False)

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(True)

    def forward(self, signals, ages, sexes):
        # signals: (B,12,1000)  ages: (B,)  sexes: (B,)
        feats = self.backbone(signals, ages, sexes)   # (B,768)
        return self.head(feats).squeeze(-1)           # (B,)


model = ViT1DClassifier().to(device)

# Load ST-MEM pretrained weights
if STMEM_CKPT.exists():
    model.backbone.load_stmem_pretrained(str(STMEM_CKPT))
    print("\n✓ ST-MEM weights loaded")
else:
    print(f"\n⚠  {STMEM_CKPT} not found — training from scratch")

n_all  = sum(p.numel() for p in model.parameters()) / 1e6
n_head = sum(p.numel() for p in model.head.parameters()) / 1e6
print(f"  Total: {n_all:.1f} M  |  Head: {n_head:.2f} M")

# Sanity check
with torch.no_grad():
    sig  = torch.zeros(2, 12, 1000, device=device)
    age  = torch.zeros(2, device=device)
    sex  = torch.zeros(2, device=device)
    out  = model(sig, age, sex)
    assert out.shape == (2,), f"Unexpected: {out.shape}"
print("  Forward pass: OK")

# AsymmetricBCE — SAME as hybrid
criterion = AsymmetricBCELoss(gamma_pos=0.0, gamma_neg=2.0, pos_weight=10.0)
scaler    = GradScaler()


In [ ]:
# ── Iteration-based training (mirrors per_2d_fold.ipynb exactly) ─────────────

def make_infinite(loader):
    while True: yield from loader


def validate(loader, n_perms=1000):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            sigs   = batch['signal'].to(device, non_blocking=True)
            ages   = batch['age'].to(device,    non_blocking=True)
            sexes  = batch['sex'].to(device,    non_blocking=True)
            labels = batch['label_hard']
            with autocast(device_type='cuda'):
                probs = torch.sigmoid(model(sigs, ages, sexes)).cpu().float().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.tolist())
    probs_arr  = np.array(all_probs,  dtype=np.float64)
    labels_arr = np.array(all_labels, dtype=np.int32)
    tpr5  = compute_challenge_score(labels_arr, probs_arr, num_permutations=n_perms)
    from sklearn.metrics import roc_auc_score, average_precision_score
    auroc = roc_auc_score(labels_arr, probs_arr)
    auprc = average_precision_score(labels_arr, probs_arr)
    model.train()
    return tpr5, auroc, auprc, probs_arr, labels_arr


def run_phase(phase_iters, lr, freeze_backbone, phase_name):
    if freeze_backbone:
        model.freeze_backbone()
        params = [p for p in model.parameters() if p.requires_grad]
    else:
        model.unfreeze_backbone()
        params = model.parameters()

    opt   = AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=phase_iters, eta_min=lr/20)

    best_tpr5   = 0.0
    train_iter  = make_infinite(train_loader)
    accum_loss  = 0.0
    global_step = 0

    model.train()
    opt.zero_grad(set_to_none=True)
    pbar = tqdm(total=phase_iters, desc=phase_name, ncols=90)

    for accum_step in range(phase_iters * GRAD_ACCUM):
        batch  = next(train_iter)
        sigs   = batch['signal'].to(device, non_blocking=True)
        ages   = batch['age'].to(device,    non_blocking=True)
        sexes  = batch['sex'].to(device,    non_blocking=True)
        labels = batch['label'].to(device,  non_blocking=True)

        with autocast(device_type='cuda'):
            logits = model(sigs, ages, sexes)
            loss   = criterion(logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()
        accum_loss += loss.item() * GRAD_ACCUM

        if (accum_step + 1) % GRAD_ACCUM == 0:
            global_step += 1
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad(set_to_none=True)
            pbar.set_postfix(loss=f"{accum_loss/global_step:.4f}", step=global_step)
            pbar.update(1)

            if global_step % VAL_EVERY == 0 or global_step == phase_iters:
                tpr5, auroc, auprc, _, _ = validate(val_loader, n_perms=1000)
                flag = ""
                if tpr5 > best_tpr5:
                    best_tpr5 = tpr5
                    torch.save(model.state_dict(), BEST_PATH)
                    flag = "  ← BEST"
                tqdm.write(f"  [{phase_name}] iter {global_step:>6}  "
                           f"TPR@5%={tpr5:.4f}  AUROC={auroc:.4f}  AUPRC={auprc:.4f}{flag}")

    pbar.close()
    return best_tpr5


print("✓ Training utilities ready")


In [ ]:
# ── PHASE 1 ── head + demo encoder  (transformer frozen) ───────────────────
print("\n" + "="*60)
print(f"PHASE 1 — {PHASE1_ITERS} iters  |  frozen backbone  |  LR={PHASE1_LR}")
print("="*60)

t0 = time.time()
best_p1 = run_phase(PHASE1_ITERS, PHASE1_LR, freeze_backbone=True, phase_name="P1")
print(f"\nPhase 1 done  ({(time.time()-t0)/60:.0f} min)  best TPR@5% = {best_p1:.4f}")


In [ ]:
# ── PHASE 2 ── full fine-tune ────────────────────────────────────────────────
print("\n" + "="*60)
print(f"PHASE 2 — {PHASE2_ITERS} iters  |  full fine-tune  |  LR={PHASE2_LR}")
print("="*60)

t0 = time.time()
best_p2 = run_phase(PHASE2_ITERS, PHASE2_LR, freeze_backbone=False, phase_name="P2")
print(f"\nPhase 2 done  ({(time.time()-t0)/60:.0f} min)  best TPR@5% = {best_p2:.4f}")
print(f"\nOverall best TPR@5% = {max(best_p1, best_p2):.4f}")


In [ ]:
# ── FINAL EVAL ───────────────────────────────────────────────────────────────
model.load_state_dict(torch.load(BEST_PATH, map_location=device))
tpr5, auroc, auprc, probs_arr, labels_arr = validate(val_loader, n_perms=10_000)
n_pos   = int(labels_arr.sum())
n_total = len(labels_arr)

print("\n" + "="*60)
print(f"FOLD {FOLD}  1D-only  —  Final Results")
print("="*60)
print(f"  TPR@5%FPR  : {tpr5:.4f}")
print(f"  AUROC      : {auroc:.4f}")
print(f"  AUPRC      : {auprc:.4f}")
print(f"  N pos/total: {n_pos} / {n_total}")

pd.DataFrame([{
    "tpr_5pct": tpr5, "auroc": auroc, "auprc": auprc,
    "using_official": True, "num_permutations": 10000,
    "n_pos": n_pos, "n_total": n_total,
    "fold": FOLD, "quick_test": False,
}]).to_csv(OUT_CSV, index=False)
print(f"\n✓ Saved: {OUT_CSV}")


In [ ]:
# ── THREE-WAY COMPARISON TABLE ───────────────────────────────────────────────
print(f"\nAblation Study  —  Fold {FOLD}  (same val set for all three)")
print(f"{'─'*62}")
print(f"{'Pathway':<26}  {'TPR@5%':>8}  {'AUROC':>8}  {'AUPRC':>8}  Note")
print(f"{'─'*62}")

# 1D only (just computed)
print(f"{'1D only  (ST-MEM + head)':<26}  {tpr5:>8.4f}  {auroc:>8.4f}  {auprc:>8.4f}")

# 2D only
csv_2d = CKPT_DIR / f"fold{FOLD}_2d_results.csv"
if csv_2d.exists():
    r2 = pd.read_csv(csv_2d).iloc[0]
    print(f"{'2D only  (MAE + head)':<26}  {r2.tpr_5pct:>8.4f}  {r2.auroc:>8.4f}  {r2.auprc:>8.4f}")
else:
    print(f"{'2D only':<26}  {'run per_2d_fold.ipynb first':>26}")

# Hybrid
hybrid_csv = CKPT_DIR / f"fold{FOLD}_results.csv"
if hybrid_csv.exists():
    h = pd.read_csv(hybrid_csv).iloc[0]
    print(f"{'Hybrid  (1D + 2D + REPA)':<26}  {h.tpr_5pct:>8.4f}  {h.auroc:>8.4f}  {h.auprc:>8.4f}  ← baseline")

print(f"{'─'*62}")
print("\nInterpretation guide:")
print("  Hybrid > 1D only  → 2D pathway adds value")
print("  Hybrid > 2D only  → 1D pathway adds value")
print("  Hybrid > both     → REPA alignment + fusion also contributes")
